# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for exploring the FAIR² protein abundance dataset with [mlcroissant](https://mlcommons.org/croissant/). You will load, preview, and interactively process experimental data described by the Croissant schema. All entities (record sets, fields, columns) are referenced by their `@id` fields as per best practice.

### Dataset Source
The dataset is provided by Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata fields
meta = dataset.metadata

print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {', '.join(meta.keywords) if hasattr(meta, 'keywords') else 'N/A'}")

## 2. Data Overview
List the available record sets, their `@id`s, and the fields available in each record set.

**Note**: In this Croissant file, the main record set containing protein-level data is typically listed under `recordSet` in the metadata, but some Croissant datasets may use `record_sets` or similar. We'll handle both situations for generality.

In [ ]:
from pprint import pprint

# Croissant allows introspecting record sets by their @id
def list_record_sets(meta):
    record_sets = []
    if hasattr(meta, 'recordSet') and meta.recordSet:
        # In Croissant v1, recordSet can be a list of objects with @id
        for r in meta.recordSet:
            # r can be a string (@id) or dict
            if isinstance(r, dict) and '@id' in r:
                record_sets.append(r['@id'])
            elif isinstance(r, str):
                record_sets.append(r)
    return record_sets

record_set_ids = list_record_sets(meta)
if not record_set_ids:
    print("No record sets listed in metadata. The Croissant may refer to default or implicit tables (check documentation).")
else:
    print("Available record sets:")
    for rs_id in record_set_ids:
        print(f"- {rs_id}")

# For demonstration, let's enumerate any fields in the first record set if present
if record_set_ids:
    rs_id = record_set_ids[0]
    record_set = dataset.metadata.by_id(rs_id)
    print(f"\nFields for record set {rs_id}:")
    if hasattr(record_set, 'field') and record_set.field:
        for f in record_set.field:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
            field_obj = dataset.metadata.by_id(field_id)
            print(f"- Field: {field_obj.name} (@id: {field_id}, type: {getattr(field_obj, 'dataType', 'N/A')})")
    else:
        print("(No fields declared on this record set; check documentation)")

## 3. Data Extraction
Load records from a specific record set as a DataFrame for analysis. Reference the record set `@id` and field `@id`s from the previous step.

_For this dataset, if `meta.recordSet` is empty, we will attempt to load the data using mlcroissant's automatic record set detection (i.e., load the first available record set)._

In [ ]:
# Will load the first record set found, fallback to listing all possible tables
if record_set_ids:
    chosen_record_set = record_set_ids[0]
else:
    # Some datasets only have one table; mlcroissant will often infer it
    # We'll introspect package structure to try to find main data file
    # For this dataset, let's try common protein table names (may need manual check)
    # In absence of explicit @ids, let dataset.records() use default
    chosen_record_set = None

# Display which record set we're loading
print(f"Loading record set: {chosen_record_set if chosen_record_set else '(default)'}")

# Load all records into a DataFrame
records = list(dataset.records(record_set=chosen_record_set) if chosen_record_set else dataset.records())
df = pd.DataFrame(records)
# Store in dict for template compatibility
dataframes = {chosen_record_set if chosen_record_set else 'default': df}

print(f"Columns in DataFrame: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by values of a numeric field, normalizing, and grouping.

_We'll use a typical protein quantification field (e.g., 'abundance', 'Coverage', or 'MW') if available and reference by column name (should correspond to the field's `@id` or label in your Croissant schema)._

In [ ]:
# Choose a numeric field for analysis (modify as appropriate for your dataset)
numeric_fields = [col for col in df.columns if col.lower() in ['abundance', 'coverage', 'mw', 'peptide_count']]
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    # Fallback: pick any float/integer-like field
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.empty else None

print(f"Using numeric field for EDA: {numeric_field}")

threshold = df[numeric_field].mean() if numeric_field else 0
print(f"Threshold (mean of {numeric_field}) = {threshold:.2f}")

# Filter rows above threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a potential categorical field (e.g., 'description', 'Group', etc.)
group_fields = [col for col in df.columns if col.lower() in ['description', 'group', 'sample']]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and grouping if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(data=df, x=numeric_field, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group, if possible
if group_fields:
    group_field = group_fields[0]
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook provided an exploration of the FAIR² protein dataset using `mlcroissant`.

**Key steps:**
- Loaded metadata and records using Croissant.
- Enumerated available record sets and fields by `@id`.
- Loaded tabular data and performed simple numerical EDA with filtering and normalization.
- Visualized data distributions.

The dataset describes protein abundance in human extracellular vesicles. For detailed analysis, consult the [dataset's Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and documentation for field semantics and best practices.